# Machine Learning - Clustering

Clustering describes a class of methods belonging to *unsupervised learning*. This means that there is no label to be predicted, which would make it *supervised learning*. Rather, we will need some way to describe *similarity* between data, since the idea of clustering is to find groups of data, where the data within the groups is similar, while the groups themselves are well distinguished and thus dissimilar.

This notebook is based on Chapter 11 of [Python Machine Learning](https://github.com/rasbt/python-machine-learning-book-3rd-edition) book.

### Overview

- [Hierarchical Clustering](#Hierarchical-Clustering)
  - [Performing hierarchical clustering on a distance matrix](#Performing-hierarchical-clustering-on-a-distance-matrix)
  - [Advanced: Attaching dendrograms to a heat map](#Advanced:-Attaching-dendrograms-to-a-heat-map)
  - [Applying agglomerative clustering via scikit-learn](#Applying-agglomerative-clustering-via-scikit-learn)
  - [Visualizing using plotly](#Visualizing-using-plotly)

- [k-means Clustering](#k-means-Clustering)
  - [Using the elbow method to find the optimal number of clusters](#Using-the-elbow-method-to-find-the-optimal-number-of-clusters)
  - [Quantifying the quality of clustering via silhouette plots](#Quantifying-the-quality-of-clustering--via-silhouette-plots)
- [Locating regions of high density via DBSCAN](#Locating-regions-of-high-density-via-DBSCAN)

First, all the neccessary imports:

In [ ]:
# if you get a warning in windows complaining about a potential memory leak with k-means, you can disable multithreading:
import os
os.environ["OMP_NUM_THREADS"] = '1'
# (seems not necessary on Linux)

import pandas as pd
import numpy as np

from scipy.spatial.distance import pdist,squareform # scientific python (scipy), used for distance computation -- conda install scipy
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import dendrogram

import matplotlib.pyplot as plt 
%matplotlib inline

import plotly.graph_objs as go # plotly is another plotting library (as matplotlib, seaborn), which allows interactive plots (conda install plotly)
import plotly.offline as pyo

# scikit-learn; this is *the* ML library for us (conda install sklearn ?)
from sklearn.cluster import AgglomerativeClustering 
from sklearn.cluster import KMeans
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import MinMaxScaler # remember "normalization" from last weeks lecture?
from sklearn.metrics import silhouette_samples

# Hierarchical Clustering


Advantages:
* number of clusters does not have to be specified beforehand
* visualization via [dendrograms](https://www.displayr.com/what-is-dendrogram/) is possible

Two main approaches:
* *divisive*, i.e. starting with all samples in one cluster and doing binary splits until there is a cluster for each sample (very high computational efford)
* *agglomerative*, i.e. starting with one cluster for each sample and combining two clusters until there is one cluster containing all samples

Suitable number of clusters is subsequently chosen by looking at the results (possibly using dendrograms).

**Criterion for cluster combination**: *linkage* in variants *single, complete, average, Ward's* or others;
then combine those clusters which have the minimum distance according to linkage chosen.

In [ ]:
from IPython.display import Image
Image(filename='./images/11_05.png', width=400) 

**Agglomerative Clustering**

Iterative procedure as follows:
1. Compute the distance matrix of all samples.
2. Represent each data point as a singleton cluster.
3. Merge the two closest clusters based on the distance of the most dissimilar(distant) members.
4. Update the distance matrix.
5. Repeat steps 2 to 4 until one single cluster remains.

*Example* Random generation of 10 random samples ID_0, ..., ID_9 with three features X, Y, Z (thus 3 dimensional data):

In [ ]:
np.random.seed(123) # we always set the random seed to something static such that every (random) run fo the code
                    # which involves randomness returns same results here in the lecture. 
                    # This is not really helpful in "production" settings.

variables = ['X', 'Y', 'Z']
itemlabels = [] 
for i in range(10): # 'ID_0',...,'ID_9',
    itemlabels.append("ID_"+str(i))

X = np.random.random_sample([5,3])*3
X2 = np.random.random_sample([5,3])*10
X = np.append(X,X2,axis=0)
df = pd.DataFrame(X, columns=variables, index=itemlabels)
df

## Performing hierarchical clustering on a distance matrix

Computation of the distance matrix. In this subsection we will use [Scientific Python](https://scipy.org/) `scipy`, later switching to `scikit-learn`.

In [ ]:
# using scipy
row_pdist = pdist(df, metric='euclidean')
row_dist = pd.DataFrame(squareform(row_pdist), columns=itemlabels, index=itemlabels)
row_dist

Compare this to the output of <tt>pdist</tt>:

In [ ]:
row_pdist

Since the distance matrix is symmetric with zeroes on the diagonal (cf. dataframe above), we really only need the upper triangular values, which is the output of pdist()-function

*Computing the **linkage matrix**:* 

We can either pass a condensed distance matrix (upper triangular) from the `pdist` function, or we can pass the "original" data array and define the `metric='euclidean'` argument in `linkage`. However, we should **not** pass the squareform distance matrix, which would yield different distance values although the overall clustering could be the same.

In [ ]:
# 1. incorrect approach: Squareform distance matrix

row_clusters = linkage(row_dist, method='complete', metric='euclidean')
df1 = pd.DataFrame(row_clusters, 
             columns=['ID 1', 'ID 2', 'distance', 'items in cluster'],
             index=['cluster %d' %(i+1) for i in range(row_clusters.shape[0])])
df1['ID 1'] = df1['ID 1'].astype(int)
df1['ID 2'] = df1['ID 2'].astype(int)
df1['items in cluster'] = df1['items in cluster'].astype(int)
df1

In [ ]:
# 2. correct approach: Condensed distance matrix

row_clusters = linkage(pdist(df, metric='euclidean'), method='complete')
df2 = pd.DataFrame(row_clusters, 
             columns=['ID 1', 'ID 2', 'distance', 'items in cluster'],
             index=['cluster %d' %(i+1) for i in range(row_clusters.shape[0])])
df2['ID 1'] = df2['ID 1'].astype(int)
df2['ID 2'] = df2['ID 2'].astype(int)
df2['items in cluster'] = df2['items in cluster'].astype(int)
df2

In [ ]:
# 3. correct approach: Input sample matrix, distance matrix will be computed by "linkage" method.

row_clusters = linkage(df.values, method='complete', metric='euclidean')
df3 = pd.DataFrame(row_clusters, 
             columns=['ID 1', 'ID 2', 'distance', 'items in cluster'],
             index=['cluster %d' %(i+1) for i in range(row_clusters.shape[0])])
df3['ID 1'] = df3['ID 1'].astype(int)
df3['ID 2'] = df3['ID 2'].astype(int)
df3['items in cluster'] = df3['items in cluster'].astype(int)
df3

The output shows the how the points/clusters are merged, i.e. in the first line the merge of ID_0 and ID_4. The result is a new cluster with ID_10.

In [ ]:
# make dendrogram black (part 1/2)
#from scipy.cluster.hierarchy import set_link_color_palette
#set_link_color_palette(['black'])

row_dendr = dendrogram(row_clusters, 
                       labels=itemlabels,
                       # make dendrogram black (part 2/2)
                       # color_threshold=np.inf
                       )
plt.tight_layout()
plt.ylabel('Euclidean distance')
plt.show()

<br>

Now, we could possibly cut the dendrogram such that we get two clusters as a result (seems reasonable number here)

## Advanced: Attaching dendrograms to a heat map (not exam relevant)

In [ ]:
# plot row dendrogram
fig = plt.figure(figsize=(8,8), facecolor='white')
axd = fig.add_axes([0.2,0.1,0.2,0.6])
# note: for matplotlib < v1.5.1, please use orientation='right'
row_dendr = dendrogram(row_clusters, orientation='left')

# reorder data with respect to clustering
df_rowclust = df.iloc[row_dendr['leaves'][::-1]]

axd.set_yticks([]) # remove cluster numbers from dendro
# remove axes spines from dendrogram
for i in axd.spines.values():
       i.set_visible(False)

# plot heatmap
axm = fig.add_axes([0.23,0.1,0.6,0.6]) # x-pos, y-pos, width, height
cax = axm.matshow(df_rowclust, interpolation='nearest', cmap='hot_r')
fig.colorbar(cax)
ylab = list(df_rowclust.index)
xlab = list(df_rowclust.columns)
axm.set_xticks(np.arange(len(xlab)))
axm.set_yticks(np.arange(len(ylab)))
axm.set_xticklabels(xlab)
axm.set_yticklabels(ylab)

# plt.savefig('./figures/heatmap.png', dpi=300)
plt.show()

This type of visualization makes the *similarity* of the items in the cluster visible, also for each dimension. 

## Applying agglomerative clustering via scikit-learn

Now it is time to switch to the **scikit-learn** library, AgglomerativeClustering has already been imported from it at the top.

In [ ]:
ac = AgglomerativeClustering(n_clusters=None, metric='euclidean', 
                             linkage='complete', distance_threshold=8.0)
labels = ac.fit_predict(X)
print('Cluster labels: %s' % labels)

Please have a look in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) about `n_clust`and `distance_threshold`.


In order to plot a dendrogram from this, we can have a look at the relevant suggested [wrapper function](https://scikit-learn.org/stable/auto_examples/cluster/plot_agglomerative_dendrogram.html) from sklearn and adapt it a bit. Please note that sklearn has no functionality to plot a dendrogram:

In [ ]:
def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)

plt.title('Hierarchical Clustering Dendrogram')
plot_dendrogram(ac, truncate_mode="level", labels=itemlabels)
plt.show()

This yields the same results as above, which is no wonder since we are using the `scipy` dendrogram plotting functionality in the wrapper function

## Visualizing using plotly

Finally, let's visualize our two clusters in a 3D plot (lucky us we have 3D data only). For that purpose we use an *interactive* plot from plotly:


In [ ]:
# plotly has already been imported (pyo -> plotly.offline)
trace1 = go.Scatter3d(
    x= X[:,0],
    y= X[:,1],
    z= X[:,2],
    mode='markers',
     marker=dict(
        color = labels, 
        size= 10,
        line=dict(
            color= labels,
            width= 12
        ),
        opacity=0.8
     )
)
data = [trace1]
layout = go.Layout(
    title= 'Clusters from Agglomerative Clustering',
    height=500,
    scene = dict(
            xaxis = dict(title  = 'X'),
            yaxis = dict(title  = 'Y'),
            zaxis = dict(title  = 'Z')
        )
)
fig = go.Figure(data=data, layout=layout)
pyo.init_notebook_mode()
pyo.iplot(fig)

# k-means Clustering

*k-means* is a well-known method for clustering, which is why we will take a closer look here. It is very popular, because compared to hierarchical clustering it is still computationally very efficient.

In k-means, the clusters are represented by their **centroid**s (the *average*) \[or their **medoid**s (most frequently occuring point incase of categorical data)\]. Therefore, k-means belongs to the class of prototype-based clustering algorithms. The algorithm turns out to be very well suited for *spherical* clusters. There is one major challenge in applying it: finding $k$, i.e. the number of clusters, which has to be specified beforehand. A poorly chosen $k$ might result in far from optimal performance, but we will see later that there are ways of (semi-)automating the choice of $k$.

To illustrate how k-means works, we will look at an example using two-dimensional data. However, the algorithm can be applied in higher dimensions just the same. We start by creating 150 randomly generated points belonging to three regions where the individual points are grouped closer together.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=150, 
                  n_features=2, 
                  centers=3, 
                  cluster_std=0.5, 
                  shuffle=True, 
                  random_state=2) # random_state sets the random seed for this sklearn method to get deterministic results.
                                  # You will see that more often in all of my notebooks. 

Let us look at the result in a *scatter plot*.

In [ ]:
plt.scatter(X[:,0], X[:,1], c='grey', marker='o', s=50)
plt.grid()
plt.tight_layout()
plt.show()

Please remember that we need to **normailze** data **always** before applying k-means (or any clustering algorithm in general). You might have already observed the different scales on the axes.

In [ ]:
# sklearn.preprocessing.MinMaxScaler already imported
scaler = MinMaxScaler()
scaler.fit(X)
X_norm = scaler.transform(X) # alternatively there is a short "fit_transform" method combing this line and the one above.

# plot again..
plt.scatter(X_norm[:,0], X_norm[:,1], c='grey', marker='o', s=50)
plt.grid()
plt.tight_layout()
plt.show()

**Algorithm**:
1. Initialization - Randomly pick $k$ centroids from the sample points as initial cluster centers.
2. Assign each sample to the nearest centroid $\mu^{(j)} , j ∈ { 1, \dots , k }$ .
3. Compute the new centroids, e.g. "move" the centroid to the center of the samples that were assigned to it.
4. Repeat steps 2 and 3 until the cluster assignments do not change or a user-defined tolerance or a maximum number of iterations is reached.

*How to we measure similarity*? 

* similarity $:=$ *opposite of distance*
* for numerical features use **squared Euclidean distance**:
  sample points $x, y\in \mathbb{R}^m$
$$
d ( \mathbf{x} , \mathbf{y} ) = \sum_{f=1}^m ( x_f − y_f )^2 = \left\lVert{\mathbf{x} − \mathbf{y}}\right\rVert_2^2
$$
* Above: $f$ refers to $f$-th dimension (feature column) of the  $x$ and $y$. 

$ \longrightarrow $ k-means algorithm is **optimization problem**, i.e. *iterative approach* for minimizing the **within-cluster sum of squared errors (SSE)**:
$$
SSE = \sum_{i=1}^n\sum_{j=1}^k w^{ ( i, j ) }\left\lVert{\mathbf{x}^{(i)} - \mathbf{\mu}^{(j)}}\right\rVert_2^2
$$

* $\mathbf{\mu}^{(j)}$ is centroid for cluster $j$
* weights $w$: 
$$ w^{ ( i, j )} = \begin{cases}1 &  x^{ ( i )}  \in C_j\\ 0 & \text{otherwise}\end{cases}$$

Next, we use the simple k-means to generate the clusters and predict the cluster labels for the individual samples.

In [ ]:
# sklearn.cluster.KMeans already imported at the top
km = KMeans(n_clusters=3, 
            init='random', # or put "k-means++" here (cf. lecture slides; try it!)
            n_init=10, # cf. lecture slides, running it 10 times and selecting the "best" result
            max_iter=300, # max_iter and tol refer to stopping critera.
            tol=1e-04,
            random_state=0)
y_km = km.fit_predict(X_norm) # fitting and predicting at the same time...
centroids_norm = km.cluster_centers_
print("Cluster belongings:")
print(y_km)
print("Centroids:")
print(centroids_norm)

Do you remeber the *fit-predict* paradigm? Fit was for "training" and predict for "testing". However, in unsupervised learning we do not need to separate this and a shortcut method is called `fit_predict`.

An then we can visualize the results (note that we are using X to show the unscaled data):

In [ ]:
# Above we got the centroids in the normalized X. If we want to plot them correctly, we need to invert the transformation..
centroids = scaler.inverse_transform(centroids_norm)

plt.scatter(X[y_km==0,0], 
            X[y_km==0,1], 
            s=50, 
            c='lightgreen', 
            marker='s', 
            label='cluster 1')
plt.scatter(X[y_km==1,0], 
            X[y_km==1,1], 
            s=50, 
            c='orange', 
            marker='o', 
            label='cluster 2')
plt.scatter(X[y_km==2,0], 
            X[y_km==2,1], 
            s=50, 
            c='lightblue', 
            marker='v', 
            label='cluster 3')
plt.scatter(centroids[:,0], 
            centroids[:,1], 
            s=250, 
            marker='*', 
            c='red', 
            label='centroids')
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig('./figures/centroids.png', dpi=300)
plt.show()

<br>

## Using the elbow method to find the optimal number of clusters 

How did I know to set k=3 in the example above? Well, because this how I generated the data. On real-world data, the $k$ is not known at all.
Please remeber this is unsupervised learning approach and there are also no labels to check the result by computing a performance measure how correct the cluster assignment is.

So we can measure the quality of the result *intrinsically*;
for k-means we can compute the SSE to the centroid or **cluster inertia** called *distortion*:

In [ ]:
print('Distortion: %.2f' % km.inertia_)

**Elbow-Idea**
The more clusters we have, the less distortion we will get until there is no distortion at all if we have one cluster for each sample. Obviously, the latter makes no sense and having $k=1$ does not either. So, we will look for a $k$ from which on the decrease in distortion will be noticeably less than it was up until that point.

So, we can try out a few $k$ and look at the plot, which should give us a good indication for a suitable $k$ which will provide a good trade-off between cluster density and amount of clusters.

In [ ]:
distortions = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, 
                init='k-means++', # this can be left out, as it is the default
                n_init=10, 
                max_iter=300, 
                random_state=0)
    km.fit(X_norm)
    distortions.append(km.inertia_)
plt.plot(range(1,11), distortions , marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Distortion')
plt.tight_layout()
#plt.savefig('./figures/elbow.png', dpi=300)
plt.show()
print(distortions)

Obviously, as expected, $k=3$ is the number of clusters to use here.

## Quantifying the quality of clustering  via silhouette plots

1. *cohesion* $a^{(i)}$: average distance between sample $\mathbf{x}^{(i)}$ and all other points in the same cluster (similarity within cluster)
2. *separation* $b^{(i)}$: average distance between sample $\mathbf{x}^{(i)}$ and all samples in the nearest cluster (dissimilarity to other clusters)
3. *silhouette* $s^{(i)}$:
   $$
     s^{(i)} := \frac{b^{(i)} - a^{(i)}}{\text{max}\{b^{(i)},a^{(i)}\}}\quad\in[-1,1]
   $$
   
Ideally, $b^{(i)}>>a^{(i)} \Rightarrow s^{(i)}\approx 1$

`silhouette_samples` available in scikit-learn (<tt>metric</tt> module)!

In [ ]:
from matplotlib import cm # a colormap

km = KMeans(n_clusters=3, 
            init='k-means++', 
            n_init=10, 
            max_iter=300,
            tol=1e-04,
            random_state=0)
y_km = km.fit_predict(X_norm)

cluster_labels = np.unique(y_km)
n_clusters = cluster_labels.shape[0]
silhouette_vals = silhouette_samples(X_norm, y_km, metric='euclidean')
y_ax_lower, y_ax_upper = 0, 0
yticks = []
for i, c in enumerate(cluster_labels):
    c_silhouette_vals = silhouette_vals[y_km==c]
    c_silhouette_vals.sort()
    y_ax_upper += len(c_silhouette_vals)
    color = cm.jet(i / n_clusters)
    plt.barh(range(y_ax_lower, y_ax_upper), c_silhouette_vals, height=1.0, 
            edgecolor='none', color=color)

    yticks.append((y_ax_lower + y_ax_upper) / 2)
    y_ax_lower += len(c_silhouette_vals)
    
silhouette_avg = np.mean(silhouette_vals)
plt.axvline(silhouette_avg, color="red", linestyle="--") 

plt.yticks(yticks, cluster_labels + 1)
plt.ylabel('Cluster')
plt.xlabel('Silhouette coefficient')

plt.tight_layout()
# plt.savefig('./figures/silhouette.png', dpi=300)
plt.show()

Nice, no silhouette value is even close to zero! Thus, three clusters seems to do it.

### Comparison to "bad" choice of $k$:

In [ ]:
km = KMeans(n_clusters=2, 
            init='k-means++', 
            n_init=10, 
            max_iter=300,
            tol=1e-04,
            random_state=0)
y_km = km.fit_predict(X_norm)

plt.scatter(X[y_km==0,0], 
            X[y_km==0,1], 
            s=50, 
            c='lightgreen', 
            marker='s', 
            label='cluster 1')
plt.scatter(X[y_km==1,0], 
            X[y_km==1,1], 
            s=50, 
            c='orange', 
            marker='o', 
            label='cluster 2')
centroids=scaler.inverse_transform(km.cluster_centers_)
plt.scatter(centroids[:,0], centroids[:,1], s=250, marker='*', c='red', label='centroids')
plt.legend()
plt.grid()
plt.tight_layout()
#plt.savefig('./figures/centroids_poor.png', dpi=300)
plt.show()

In [ ]:
cluster_labels = np.unique(y_km)
n_clusters = cluster_labels.shape[0]
silhouette_vals = silhouette_samples(X_norm, y_km, metric='euclidean')
y_ax_lower, y_ax_upper = 0, 0
yticks = []
for i, c in enumerate(cluster_labels):
    c_silhouette_vals = silhouette_vals[y_km==c]
    c_silhouette_vals.sort()
    y_ax_upper += len(c_silhouette_vals)
    color = cm.jet(i / n_clusters)
    plt.barh(range(y_ax_lower, y_ax_upper), c_silhouette_vals, height=1.0, 
            edgecolor='none', color=color)

    yticks.append((y_ax_lower + y_ax_upper) / 2)
    y_ax_lower += len(c_silhouette_vals)
    
silhouette_avg = np.mean(silhouette_vals)
plt.axvline(silhouette_avg, color="red", linestyle="--") 

plt.yticks(yticks, cluster_labels + 1)
plt.ylabel('Cluster')
plt.xlabel('Silhouette coefficient')

plt.tight_layout()
# plt.savefig('./figures/silhouette_poor.png', dpi=300)
plt.show()

For cluster 1, there are samples whose cohesion is more or less the same as the separation $\rightarrow$ not good!

**Note**: This can be visualized even when the samples have more than two dimensions (where the scatter plots will not make a lot of sense)

# Locating regions of high density via DBSCAN

Density-based Spatial Clustering of Applications with Noise (DBSCAN). We did not discuss this algorithm in detail in the lecture, but maybe worth a try! In contrast to the algorithms we used so far, DBSCAN also models *noise*, which is not assigned to a cluster.

+ *density* = number of points within specified radius $\varepsilon$
+ *core point*: at least a specified number $n_{\text{min}}$ of neighboring points fall within the specified radius $\varepsilon$
+ *border point*: has fewer neighbors than $n_{\text{min}}$ within but lies within the $\varepsilon$ radius of a core point
+ *noise point*: points that are neither core nor border points


In [ ]:
Image(filename='./images/11_11.png', width=500) 

**DBSCAN Algorithm**

1. Label points as core, border or noise points
2. Form a separate cluster for each core point or a connected group of core points (core points are connected if they are not further away than $\varepsilon$ ).
3. Assign each border point to the cluster of its corresponding core point.

- Advantage of DBSCAN is mainly that shape of clusters does *not* have to be *spherical*
- noise points will not be clustered $\rightarrow$ outliers

In [ ]:
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.05, random_state=0)
X = np.append(X,np.array([[-0.5,-1]]),axis=0) # adding an outlier
plt.scatter(X[:,0], X[:,1])
plt.tight_layout()
#plt.savefig('./figures/moons.png', dpi=300)
plt.show()

*Visually*, two clusters are recognized immediately. However ....

#### K-means and hierarchical clustering:

In [ ]:
f, (ax1, ax2) = plt.subplots(1, 2, figsize=(8,3))

km = KMeans(n_clusters=2, n_init=10, random_state=42)
y_km = km.fit_predict(X)
ax1.scatter(X[y_km==0,0], X[y_km==0,1], c='lightblue', marker='o', s=40, label='cluster 1')
ax1.scatter(X[y_km==1,0], X[y_km==1,1], c='red', marker='s', s=40, label='cluster 2')
ax1.set_title('K-means clustering')

ac = AgglomerativeClustering(n_clusters=2, metric='euclidean', linkage='complete') # you know what happens when changed to "single", right?
y_ac = ac.fit_predict(X)
ax2.scatter(X[y_ac==0,0], X[y_ac==0,1], c='lightblue', marker='o', s=40, label='cluster 1')
ax2.scatter(X[y_ac==1,0], X[y_ac==1,1], c='red', marker='s', s=40, label='cluster 2')
ax2.set_title('Agglomerative clustering (complete l.)')

plt.legend()
plt.tight_layout()
#plt.savefig('./figures/kmeans_and_ac.png', dpi=300)
plt.show()

Clearly, the spherical shape of the clusters based on Euclidean distance becomes apparent here. Note the outlier at the bottom!

#### Density-based clustering:

In [ ]:
# using sklearn.cluster.DBSCAN

db = DBSCAN(eps=0.2, min_samples=5, metric='euclidean')
y_db = db.fit_predict(X)

labels = db.labels_
# Identify the outliers
outliers = np.where(labels == -1)[0]

plt.scatter(X[y_db==0,0], X[y_db==0,1], c='lightblue', marker='o', s=40, label='cluster 1')
plt.scatter(X[y_db==1,0], X[y_db==1,1], c='orange', marker='s', s=40, label='cluster 2')
plt.scatter(X[outliers, 0], X[outliers, 1], c="red", marker="x", label='outlier')
plt.legend()
plt.tight_layout()
#plt.savefig('./figures/moons_dbscan.png', dpi=300)
plt.show()

DBSCAN finds the clusters a human recognizes immediately in this case and the (manually inserted) outlier

*Disadvantages of DBSCAN*

- [curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality) increases with number of features
- $\varepsilon$ and $n_{\text{min}}$ must be optimized for good results $\rightarrow$ might be hard. Try to set  $\varepsilon$ to $0.1$ or $0.5$ above and check the very poor performance!

...